In [0]:

CREATE OR REPLACE TEMP VIEW Silver_Members AS
SELECT *
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`;

In [0]:
--Verify the records
select * from silver_members;

Member_Name,Member_ID,Enrollment_Date,Last_Flight_Date,DOB,Tier_Code,Agent_Name,State,Country,Post_Code,Is_Active,Member_Type,Source_Country,Age,Stale_Member,Enrollment_Date_Status,Last_Flight_Date_Status,DOB_Status,Ingested_At
Mike,1,2022-05-11,2022-08-01,null,PLT,null,null,null,null,null,null,AUS,null,true,VALID,VALID,INVALID,2026-11-21T07:18:39.239Z
Jonnathan,2,null,2022-01-05,1997-12-13,GLD,null,null,null,null,null,null,AUS,28,true,INVALID,VALID,VALID,2026-11-21T07:18:39.239Z
Cristina,3,2022-03-12,2022-03-20,1998-03-12,SLV,null,null,null,null,null,null,AUS,28,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Vikas,1,2022-01-01,2022-06-15,1998-12-01,SLV,null,null,null,null,null,I,IND,27,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Rahul,2,2022-03-05,2022-03-10,1982-08-13,GLD,null,null,null,null,null,C,IND,44,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Sameer,3,2022-02-20,2022-02-25,1952-08-13,GLD,null,null,null,null,null,I,IND,74,true,VALID,VALID,VALID,2026-11-21T07:18:39.239Z
Sam,1,2022-06-15,2022-08-20,null,PLT,null,null,null,null,null,null,USA,null,true,VALID,VALID,MISSING,2026-11-21T07:18:39.239Z
John,2,2022-01-05,2022-01-15,null,SLV,null,null,null,null,null,null,USA,null,true,VALID,VALID,MISSING,2026-11-21T07:18:39.239Z
Mike,3,2021-12-28,2021-12-30,null,GLD,null,null,null,null,null,null,USA,null,true,VALID,VALID,MISSING,2026-11-21T07:18:39.239Z


Country specific target table

In [0]:
-- Step 9: Create latest member records

CREATE OR REPLACE TEMP VIEW latest_members AS

SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY Member_Name
            ORDER BY Enrollment_Date DESC NULLS LAST
        ) AS Record_Rank
    FROM silver_members
)
WHERE Record_Rank = 1;

In [0]:
SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    Enrollment_Date,
    Record_Rank
FROM latest_members
ORDER BY Member_Name;

Member_Name,Member_ID,Source_Country,Enrollment_Date,Record_Rank
Cristina,3,AUS,2022-03-12,1
John,2,USA,2022-01-05,1
Jonnathan,2,AUS,null,1
Mike,1,AUS,2022-05-11,1
Rahul,2,IND,2022-03-05,1
Sam,1,USA,2022-06-15,1
Sameer,3,IND,2022-02-20,1
Vikas,1,IND,2022-01-01,1


In [0]:
--  Create Australia target table

CREATE OR REPLACE TABLE
delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_Australia/`
USING DELTA
AS
SELECT *
FROM latest_members
WHERE Source_Country = 'AUS'
AND Record_Rank = 1;

num_affected_rows,num_inserted_rows


In [0]:
-- Create India target table

CREATE OR REPLACE TABLE
delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_India/`
USING DELTA
AS
SELECT *
FROM latest_members
WHERE Source_Country = 'IND'
AND Record_Rank = 1;

num_affected_rows,num_inserted_rows


In [0]:
-- Create USA target table

CREATE OR REPLACE TABLE
delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_USA/`
USING DELTA
AS
SELECT *
FROM latest_members
WHERE Source_Country = 'USA'
AND Record_Rank = 1;

num_affected_rows,num_inserted_rows


In [0]:
SELECT 'Australia' AS Country, COUNT(*) AS Total_Records
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_Australia/`

UNION ALL

SELECT 'India', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_India/`

UNION ALL

SELECT 'USA', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_USA/`;

Country,Total_Records
Australia,3
India,3
USA,2
